In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

# --- 1. Load Your Processed PIMA Data ---
print("Loading processed PIMA data...")
pima_df = pd.read_csv('../data/03_processed/pima_final.csv')

# --- 2. Data Cleaning for PIMA ---
# The PIMA dataset has a known issue where missing values are coded as 0.
# We will replace these 0s with NaN (Not a Number) so they can be imputed.
cols_to_clean = ['Fasting_Glucose', 'Diastolic_BP', 'Skin_Thickness', 'Insulin', 'BMI']
pima_df[cols_to_clean] = pima_df[cols_to_clean].replace(0, np.nan)
pima_df.dropna(subset=['Diabetes_Outcome'], inplace=True)

# --- 3. Separate Features and Target ---
X = pima_df.drop(columns=['Diabetes_Outcome'])
y = pima_df['Diabetes_Outcome']

# --- 4. Split PIMA data into Training and Testing sets ---
# We use a standard 80/20 split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"PIMA data split into {len(X_train)} training samples and {len(X_test)} testing samples.")

# --- 5. Impute Missing Values ---
# We fit the imputer on the PIMA training data only
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# --- 6. Tune and Train the XGBoost Model on PIMA Data ---
print("\nTuning XGBoost model on PIMA training data...")
xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

# Use the same hyperparameter grid as before
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

# Use GridSearchCV to find the best model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(xgb_model, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_imputed, y_train)

best_model = grid_search.best_estimator_
print(f"Best parameters found: {grid_search.best_params_}")

# --- 7. Evaluate on PIMA Test Set ---
y_pred_proba_pima = best_model.predict_proba(X_test_imputed)[:, 1]
auc_pima = roc_auc_score(y_test, y_pred_proba_pima)

print("\n--- PIMA Benchmark Result ---")
print(f"AUC on PIMA Test Set: {auc_pima:.4f}")

Loading processed PIMA data...
PIMA data split into 614 training samples and 154 testing samples.

Tuning XGBoost model on PIMA training data...
Best parameters found: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}

--- PIMA Benchmark Result ---
AUC on PIMA Test Set: 0.8215


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Import the four models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb

print("Starting Pima Indians Dataset Benchmark Analysis...")

# --- 1. Load Your Processed Pima Data ---
pima_filepath = r'C:\diabetes_prediction_project\data\03_processed\pima_final.csv'
try:
    df_pima = pd.read_csv(pima_filepath)
    print("Successfully loaded the Pima dataset.")
except FileNotFoundError:
    print(f"ERROR: The file was not found at the path: {pima_filepath}")
    exit()

# --- 2. Prepare Data ---
try:
    X = df_pima.drop('Diabetes_Outcome', axis=1)
    y = df_pima['Diabetes_Outcome']
except KeyError:
    print("ERROR: Could not find the 'Diabetes_Outcome' column.")
    exit()

# --- 3. Define Models ---
models = {
    'XGBoost (this study)': xgb.XGBClassifier(
        learning_rate=0.05, max_depth=3, n_estimators=100,
        use_label_encoder=False, eval_metric='logloss', random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        C=10.0, penalty='l2', solver='liblinear', class_weight='balanced', max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=10,
        class_weight='balanced', random_state=42
    ),
    'SVM': SVC(kernel='linear', C=0.1, probability=True, class_weight='balanced', random_state=42)
}

# --- 4. Run Cross-Validation ---
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
our_results = []

print("Running 10-fold cross-validation for each model...")
for name, model in models.items():
    pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=['roc_auc', 'accuracy', 'recall', 'f1'])
    
    our_results.append({
        'Method': name,
        'AUC': np.mean(scores['test_roc_auc']),
        'Accuracy': np.mean(scores['test_accuracy']),
        'Sensitivity': np.mean(scores['test_recall']),
        'F1': np.mean(scores['test_f1']),
        'Reference/Note': 'Our implementation (10-fold CV)'
    })
print("Cross-validation complete.")

# --- 5. Combine with Literature Benchmarks ---
literature_benchmarks = [
    {'Method': 'Tasin et al. (2023)', 'AUC': 0.840, 'Accuracy': 0.810, 'Sensitivity': 0.810, 'F1': 0.810, 'Reference/Note': 'XGBoost + ADASYN'},
    {'Method': 'Kopitar et al. (2020)', 'AUC': 0.859, 'Accuracy': np.nan, 'Sensitivity': 0.764, 'F1': np.nan, 'Reference/Note': 'Glmnet (T30)'},
    {'Method': 'Hassan et al. (2020)', 'AUC': 0.950, 'Accuracy': np.nan, 'Sensitivity': np.nan, 'F1': np.nan, 'Reference/Note': 'Ensemble method'}
]

df_our_results = pd.DataFrame(our_results)
df_literature = pd.DataFrame(literature_benchmarks)
table8_benchmark = pd.concat([df_literature, df_our_results], ignore_index=True).set_index('Method')

# --- 6. Display and Save the Final Table (CORRECTED FORMATTING) ---
print("\n\n--- Final, Publication-Quality Table 8: Pima Indians Dataset Benchmark ---")

# Define a dictionary to format ONLY the numeric columns
formatter = {
    'AUC': '{:.4f}',
    'Accuracy': '{:.4f}',
    'Sensitivity': '{:.4f}',
    'F1': '{:.4f}'
}
# Apply the specific formatter to the display style
display(table8_benchmark.style.format(formatter, na_rep='—'))

output_path = 'results/TABLE_8_pima_benchmark.csv'
table8_benchmark.to_csv(output_path)
print(f"\nEnhanced Table 8 has been successfully saved to: {output_path}")

Starting Pima Indians Dataset Benchmark Analysis...
Successfully loaded the Pima dataset.
Running 10-fold cross-validation for each model...


c:\diabetes_prediction_project\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:43:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\diabetes_prediction_project\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:43:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\diabetes_prediction_project\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:43:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\diabetes_prediction_project\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:43:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_

Cross-validation complete.


--- Final, Publication-Quality Table 8: Pima Indians Dataset Benchmark ---


,AUC,Accuracy,Sensitivity,F1,Reference/Note
Method,,,,,
Tasin et al. (2023),0.8400,0.8100,0.8100,0.8100,XGBoost + ADASYN
Kopitar et al. (2020),0.8590,—,0.7640,—,Glmnet (T30)
Hassan et al. (2020),0.9500,—,—,—,Ensemble method
XGBoost (this study),0.8411,0.7668,0.6044,0.6422,Our implementation (10-fold CV)
Logistic Regression,0.8318,0.7499,0.7204,0.6686,Our implementation (10-fold CV)
Random Forest,0.8348,0.7577,0.7574,0.6856,Our implementation (10-fold CV)
SVM,0.8305,0.7512,0.6980,0.6623,Our implementation (10-fold CV)



Enhanced Table 8 has been successfully saved to: results/TABLE_8_pima_benchmark.csv
